## Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import logging


## Load cleaned dataset

In [2]:
df = pd.read_csv('../data/processed/clean_diabetic_data.csv')

## Prevent Data Leakage

In [ ]:
# Ensure all records for a specific patient end up in either train or test
unique_patients = df['patient_nbr'].unique()
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_df = df[df['patient_nbr'].isin(train_patients)].copy()
test_df = df[df['patient_nbr'].isin(test_patients)].copy()

# No longer need patient_nbr so drop it to prevent it being trained on
train_df = train_df.drop(columns=['patient_nbr'])
test_df = test_df.drop(columns=['patient_nbr'])

## Seperate Features

In [4]:
X_train = train_df.drop(columns=['target'])
y_train = train_df['target']

X_test = test_df.drop(columns=['target'])
y_test = test_df['target']

## One-Hot Encoding

In [5]:
# Convert categorical text columns into 1s and 0s
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# Align columns to ensure both sets have the exact same features after encoding
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

## Apply SMOTE

In [6]:
print(f"Original Training Target Distribution:\n{y_train.value_counts()}")

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_encoded, y_train)

print(f"\nBalanced Training Target Distribution (After SMOTE):\n{y_train_balanced.value_counts()}")
print(f"\nTest Set Target Distribution (Unchanged):\n{y_test.value_counts()}")

Original Training Target Distribution:
target
0    70806
1     8876
Name: count, dtype: int64

Balanced Training Target Distribution (After SMOTE):
target
0    70806
1    70806
Name: count, dtype: int64

Test Set Target Distribution (Unchanged):
target
0    17517
1     2293
Name: count, dtype: int64


## Save datasets

In [7]:
X_train_balanced.to_csv('../data/processed/X_train_balanced.csv', index=False)
y_train_balanced.to_csv('../data/processed/y_train_balanced.csv', index=False)
X_test_encoded.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)